# MOT17 Fine-tuning과 추적 성능 평가 (Google Colab)

과제 요구 사항 1·3·6·7·10을 Colab GPU에서 한 번에 진행합니다.

1. MOT17 내려받기와 전처리 (train / val / test 분할)
2. **Fine-tuning 전** 성능 측정 (기본 `yolo11n.pt`, COCO 사전학습)
3. MOT17로 YOLO11n Fine-tuning
4. **Fine-tuning 후** 성능 측정과 전후 비교 → 더 좋은 검출 모델 선택
5. 추적 파라미터 튜닝 (val) → 최종 비교 (test), 개선 항목별 효과, 실패 사례 분석
6. 결과를 Google Drive에 저장 → 내 PC 프로젝트 폴더로 복사

**시작 전에**: 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하고, 위에서부터 차례로 실행하세요.
처리 속도(요구 사항 8)는 발표 PC에서 `benchmark_speed.py`로 따로 측정합니다.

## 0. GPU 확인과 Google Drive 연결

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 경로 설정 — 필요하면 여기만 바꾸세요.
DRIVE = '/content/drive/MyDrive'
OUT = f'{DRIVE}/AI_Project_Level_3_outputs'       # 학습·평가 결과를 저장할 Drive 폴더
ZIP_CACHE = f'{DRIVE}/datasets/MOT17.zip'          # MOT17.zip을 Drive에 보관해 두면 다음부터 다시 받지 않습니다.
PROJECT = '/content/AI_Project_Level_3'
EPOCHS = 30
BATCH = 32        # T4 GPU 기준. 메모리 부족(CUDA out of memory)이 나면 16으로 줄이세요.

## 1. 코드 받기와 패키지 설치

In [ ]:
import os
if not os.path.isdir(PROJECT):
    !git clone -q https://github.com/Pdint/AI_Project_Level_3.git {PROJECT}
%cd {PROJECT}
!git log --oneline -1
# GUI(PySide6)는 Colab에서 쓰지 않으므로 학습·평가에 필요한 패키지만 설치합니다.
!pip -q install ultralytics==8.3.233 scipy

## 2. MOT17 내려받기

`MOT17.zip`(약 5.5GB, 이미지와 정답 포함)을 받습니다. Drive에 이미 있으면 그것을 씁니다.
자동 다운로드가 실패하면 https://motchallenge.net/data/MOT17/ 에서 **MOT17.zip**을 직접 받아
Drive의 `datasets/MOT17.zip` 위치에 올린 뒤 이 셀을 다시 실행하세요.

In [ ]:
import os, glob
os.makedirs(os.path.dirname(ZIP_CACHE), exist_ok=True)
if not os.path.isfile(ZIP_CACHE):
    !wget -q --show-progress -O /content/MOT17.zip https://motchallenge.net/data/MOT17.zip
    !cp /content/MOT17.zip "{ZIP_CACHE}"
else:
    !cp "{ZIP_CACHE}" /content/MOT17.zip

# MOT17은 같은 영상이 DPM/FRCNN/SDP 3벌 들어 있어 train의 FRCNN만 풉니다 (전처리도 FRCNN만 사용).
!mkdir -p datasets/raw
!unzip -q -o /content/MOT17.zip '*train/*FRCNN*' -d datasets/raw
train_dirs = sorted({os.path.dirname(p) for p in glob.glob('datasets/raw/**/train/MOT17-*-FRCNN', recursive=True)})
assert train_dirs, 'MOT17 train 폴더를 찾지 못했습니다. ZIP 파일을 확인하세요.'
MOT17_TRAIN = train_dirs[0]
print(MOT17_TRAIN, sorted(os.listdir(MOT17_TRAIN)))

## 3. 전처리: train / val / test 분할 (요구 사항 1)

시퀀스마다 앞 50%는 학습, 다음 25%는 검증(튜닝), 마지막 25%는 테스트(최종 보고)에 씁니다.

In [ ]:
!rm -rf datasets/mot
!python prepare_dataset.py mot --sequences {MOT17_TRAIN} --output datasets/mot
!cat datasets/mot/splits.json | head -40

## 4. Fine-tuning 전 성능 (기본 yolo11n.pt)

같은 test 구간에서 학습 전후를 비교하기 위한 기준값입니다.

In [ ]:
!python run_mot.py datasets/mot/tracking/test --weights yolo11n.pt --trackers baseline improved --name pretrained --device 0

## 5. Fine-tuning (요구 사항 6)

YOLO11n을 MOT17 보행자로 학습합니다. 끝나면 val·test mAP가 출력되고 `best.pt` 옆에 학습 설정·성능 `best.json`이 저장됩니다.

In [ ]:
!python train_models.py mot --data datasets/mot/dataset.yaml --weights yolo11n.pt --epochs {EPOCHS} --batch {BATCH} --workers 4 --device 0

## 6. Fine-tuning 후 성능과 전후 비교

In [ ]:
BEST = f'{PROJECT}/training_runs/mot/weights/best.pt'
!python run_mot.py datasets/mot/tracking/test --weights {BEST} --trackers baseline improved --name finetuned --device 0

In [ ]:
# 개선판 추적기 기준 HOTA로 더 좋은 검출 모델을 고르고, Fine-tuning 모델이 좋으면 models/person.pt로 설치합니다.
import json, shutil
rows = json.load(open('results/comparison.json', encoding='utf-8'))
hota = {r['name']: r['HOTA'] for r in rows if r['tracker'] == 'improved' and r['name'] in ('pretrained', 'finetuned')}
print('개선판 HOTA:', hota)
if hota['finetuned'] > hota['pretrained']:
    shutil.copyfile(BEST, 'models/person.pt')
    shutil.copyfile(BEST.replace('.pt', '.json'), 'models/person.json')
    WEIGHTS = 'models/person.pt'
    print('Fine-tuning 모델이 더 좋아 models/person.pt(+ person.json)로 설치했습니다.')
else:
    WEIGHTS = 'yolo11n.pt'
    print('Fine-tuning 모델이 더 낫지 않아 기본 yolo11n.pt를 계속 씁니다. (이 결과도 보고서에 쓸 수 있습니다.)')

## 7. 추적 파라미터 튜닝 (val 구간) → 최종 비교 (test 구간)

검출 임계값·연관성 임계값·추적 유지 시간 27개 조합 중 val HOTA가 가장 높은 설정을 `configs/tuned.json`으로 저장합니다.

In [ ]:
!python tune_params.py datasets/mot/tracking/val --base improved --name tuned --weights {WEIGHTS} --device 0
!python run_mot.py datasets/mot/tracking/test --weights {WEIGHTS} --trackers baseline improved tuned --name test --device 0

## 8. 개선 항목별 효과 (ablation, 요구 사항 3)

In [ ]:
!python run_mot.py datasets/mot/tracking/test --weights {WEIGHTS} --trackers baseline baseline+appearance baseline+recover_lost baseline+reid_buffer baseline+freeze_out_of_frame --name ablation --device 0

## 9. 가림·교차 실패 사례 분석 (요구 사항 4·7)

In [ ]:
!python analyze_failures.py datasets/mot/tracking/test --trackers baseline improved --name test --images 12 --weights {WEIGHTS} --device 0
!cat results/failures/test/summary.md

## 10. 결과를 Google Drive에 저장

`AI_Project_Level_3_outputs` 폴더의 내용을 내 PC 프로젝트 폴더에 **같은 경로로** 복사하면 GUI 모델 목록, 추적기 목록(튜닝), 성능 비교 탭에 나타납니다.

In [ ]:
!mkdir -p "{OUT}/models" "{OUT}/configs" "{OUT}/training_runs"
!cp -v models/person.* "{OUT}/models/" 2>/dev/null || echo 'models/person.pt 없음 (기본 yolo11n.pt 사용)'
!cp -v configs/*.json "{OUT}/configs/"
!rsync -a --exclude cache --exclude tmp results "{OUT}/"
!rsync -a --exclude '*.cache' training_runs/mot "{OUT}/training_runs/"
!cat results/comparison.md